# Notebook 1 — Classical Machine Learning
## Dataset: Brazilian E-Commerce (Olist) — Kaggle

**What this notebook covers:**
- Data Engineering: merging 9 real-world CSV tables
- Data Cleaning: nulls, duplicates, outliers, date parsing
- Feature Engineering: RFM, delivery delay, review sentiment proxy
- EDA: distributions, correlations, category analysis
- ML Target 1 — Regression: predict `order_value`
- ML Target 2 — Classification: predict `late_delivery` (0/1)
- Models: Linear/Logistic → Tree → Ensemble → Boosting
- Evaluation, comparison, and export

**Dataset source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce  
**Download instructions:** `kaggle datasets download -d olistbr/brazilian-ecommerce`

---
## 0. Environment Setup

In [ ]:
# Install dependencies (run once)
# !pip install kaggle pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm catboost shap missingno plotly --quiet

import os, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import plotly.express as px
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

# Regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
print('All libraries loaded successfully.')

---
## 1. Data Engineering — Load & Merge 9 Tables

The Olist dataset has 9 separate CSV files representing a real e-commerce database schema:
- **orders** — master order table (status, timestamps)
- **order_items** — line items per order (product, seller, price, freight)
- **order_payments** — payment type, installments, value
- **order_reviews** — review score, comment title/message
- **customers** — customer city, state, zip
- **sellers** — seller city, state, zip
- **products** — product category, dimensions, weight
- **product_category_name_translation** — Portuguese → English
- **geolocation** — lat/lng by zip code

In [ ]:
# ── Option A: kaggle API download ─────────────────────────────────────────────
# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY']      = 'your_api_key'
# !kaggle datasets download -d olistbr/brazilian-ecommerce -p ./data --unzip

# ── Option B: direct path after manual download ───────────────────────────────
DATA_DIR = './data'   # change if needed

# ── Load all 9 tables ─────────────────────────────────────────────────────────
orders      = pd.read_csv(f'{DATA_DIR}/olist_orders_dataset.csv')
items       = pd.read_csv(f'{DATA_DIR}/olist_order_items_dataset.csv')
payments    = pd.read_csv(f'{DATA_DIR}/olist_order_payments_dataset.csv')
reviews     = pd.read_csv(f'{DATA_DIR}/olist_order_reviews_dataset.csv')
customers   = pd.read_csv(f'{DATA_DIR}/olist_customers_dataset.csv')
sellers     = pd.read_csv(f'{DATA_DIR}/olist_sellers_dataset.csv')
products    = pd.read_csv(f'{DATA_DIR}/olist_products_dataset.csv')
category_tr = pd.read_csv(f'{DATA_DIR}/product_category_name_translation.csv')
geo         = pd.read_csv(f'{DATA_DIR}/olist_geolocation_dataset.csv')

print('Shapes of all tables:')
for name, df in [('orders', orders), ('items', items), ('payments', payments),
                  ('reviews', reviews), ('customers', customers),
                  ('sellers', sellers), ('products', products),
                  ('category_tr', category_tr), ('geo', geo)]:
    print(f'  {name:20s}: {df.shape}')

In [ ]:
# ── Step 1: Aggregate items per order ─────────────────────────────────────────
# Each order can have multiple line items — aggregate to order level
items_agg = items.groupby('order_id').agg(
    num_items        = ('order_item_id', 'count'),
    total_price      = ('price', 'sum'),
    total_freight    = ('freight_value', 'sum'),
    num_sellers      = ('seller_id', 'nunique'),
    num_products     = ('product_id', 'nunique'),
).reset_index()
items_agg['order_value'] = items_agg['total_price'] + items_agg['total_freight']
print('Items aggregated:', items_agg.shape)

# ── Step 2: Aggregate payments per order ──────────────────────────────────────
payments_agg = payments.groupby('order_id').agg(
    total_payment    = ('payment_value', 'sum'),
    payment_types    = ('payment_type', 'nunique'),
    max_installments = ('payment_installments', 'max'),
    primary_payment  = ('payment_type', lambda x: x.mode()[0] if len(x) else None)
).reset_index()

# ── Step 3: Aggregate reviews per order ───────────────────────────────────────
# Take most recent review if multiple exist
reviews_agg = reviews.sort_values('review_creation_date').groupby('order_id').agg(
    review_score        = ('review_score', 'last'),
    has_review_comment  = ('review_comment_message', lambda x: x.notna().any().astype(int))
).reset_index()

# ── Step 4: Translate product categories ──────────────────────────────────────
products = products.merge(category_tr, on='product_category_name', how='left')

# ── Step 5: Get dominant product category per order ───────────────────────────
items_with_cat = items.merge(products[['product_id','product_category_name_english',
                                        'product_weight_g','product_length_cm',
                                        'product_height_cm','product_width_cm']],
                              on='product_id', how='left')
cat_per_order = items_with_cat.groupby('order_id').agg(
    main_category   = ('product_category_name_english', lambda x: x.mode()[0] if x.notna().any() else None),
    avg_weight_g    = ('product_weight_g', 'mean'),
    total_volume_cm3= ('product_length_cm', lambda x: (x * items_with_cat.loc[x.index,'product_height_cm'] * items_with_cat.loc[x.index,'product_width_cm']).sum())
).reset_index()

print('All aggregations done.')

In [ ]:
# ── Step 6: Master join — stitch all tables together ──────────────────────────
df = (orders
      .merge(customers[['customer_id','customer_city','customer_state','customer_zip_code_prefix']], on='customer_id', how='left')
      .merge(items_agg,       on='order_id', how='left')
      .merge(payments_agg,    on='order_id', how='left')
      .merge(reviews_agg,     on='order_id', how='left')
      .merge(cat_per_order,   on='order_id', how='left')
     )

print(f'Master dataframe shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

---
## 2. Data Cleaning
### 2.1 Initial inspection

In [ ]:
print('=== SHAPE ==='); print(df.shape)
print('\n=== DTYPES ==='); print(df.dtypes)
print('\n=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
print(pd.DataFrame({'count': missing[missing>0], 'pct': missing_pct[missing>0]}).sort_values('pct', ascending=False))

In [ ]:
# Visualize missingness patterns
fig, ax = plt.subplots(figsize=(14, 5))
msno.matrix(df, ax=ax, sparkline=False)
plt.title('Missing Value Matrix — white = missing', fontsize=13)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
msno.bar(df, ax=ax)
plt.title('Data Completeness per Column', fontsize=13)
plt.tight_layout()
plt.show()

### 2.2 Parse datetime columns

In [ ]:
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print('Datetime columns converted.')
print(df[datetime_cols].dtypes)

### 2.3 Handle duplicates

In [ ]:
print(f'Duplicate order_ids: {df["order_id"].duplicated().sum()}')
print(f'Fully duplicate rows: {df.duplicated().sum()}')

# Keep first occurrence if any duplicates
df = df.drop_duplicates(subset='order_id', keep='first').reset_index(drop=True)
print(f'Shape after dedup: {df.shape}')

### 2.4 Filter to delivered orders only (data quality)

In [ ]:
print('Order status distribution:')
print(df['order_status'].value_counts())

# Only keep delivered orders for full feature availability
df = df[df['order_status'] == 'delivered'].copy()
print(f'\nShape after filtering to delivered: {df.shape}')

### 2.5 Outlier detection and treatment

In [ ]:
# Inspect distributions of key numeric columns
numeric_cols = ['order_value', 'num_items', 'total_price', 'total_freight',
                'total_payment', 'max_installments', 'avg_weight_g']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(numeric_cols):
    ax = axes[i // 4][i % 4]
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='#9ecae1'))
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('')
axes[1][3].set_visible(False)
plt.suptitle('Boxplots — Outlier Detection', fontsize=13)
plt.tight_layout()
plt.show()

print('\nStats before outlier treatment:')
print(df[numeric_cols].describe().round(2))

In [ ]:
def cap_outliers_iqr(series, lower_q=0.01, upper_q=0.99):
    """Cap outliers at 1st and 99th percentile (Winsorization)."""
    lower = series.quantile(lower_q)
    upper = series.quantile(upper_q)
    return series.clip(lower, upper)

# Cap extreme values in price and freight
for col in ['order_value', 'total_price', 'total_freight', 'avg_weight_g']:
    before = df[col].describe()
    df[col] = cap_outliers_iqr(df[col])
    after  = df[col].describe()
    print(f'{col}: max {before["max"]:.1f} → {after["max"]:.1f}')

print('\nOutliers capped at 1st/99th percentile.')

### 2.6 Handle missing values

In [ ]:
# --- Numeric: fill with median ---
num_fill = ['avg_weight_g', 'total_volume_cm3', 'max_installments', 'review_score']
for col in num_fill:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'{col}: filled with median={median_val:.2f}')

# --- Categorical: fill with mode or 'unknown' ---
cat_fill = ['main_category', 'primary_payment', 'customer_state']
for col in cat_fill:
    if col in df.columns:
        mode_val = df[col].mode()[0] if df[col].notna().any() else 'unknown'
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: filled with mode="{mode_val}"')

# --- Boolean: fill with 0 ---
df['has_review_comment'] = df['has_review_comment'].fillna(0).astype(int)

print(f'\nRemaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

---
## 3. Feature Engineering

In [ ]:
# ── Delivery features ─────────────────────────────────────────────────────────
df['delivery_days_actual']    = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['delivery_days_estimated'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days
df['delivery_delay_days']     = df['delivery_days_actual'] - df['delivery_days_estimated']
df['late_delivery']           = (df['delivery_delay_days'] > 0).astype(int)  # ← CLASSIFICATION TARGET

print('Late delivery rate:', df['late_delivery'].mean().round(3))
print('Delivery days stats:')
print(df[['delivery_days_actual','delivery_days_estimated','delivery_delay_days']].describe().round(1))

In [ ]:
# ── Time features from purchase timestamp ─────────────────────────────────────
df['purchase_hour']       = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek']  = df['order_purchase_timestamp'].dt.dayofweek   # 0=Mon
df['purchase_month']      = df['order_purchase_timestamp'].dt.month
df['purchase_quarter']    = df['order_purchase_timestamp'].dt.quarter
df['is_weekend']          = (df['purchase_dayofweek'] >= 5).astype(int)
df['is_business_hours']   = df['purchase_hour'].between(9, 18).astype(int)

# ── Price features ────────────────────────────────────────────────────────────
df['freight_ratio']       = df['total_freight'] / (df['total_price'] + 1)
df['avg_item_price']      = df['total_price'] / df['num_items'].clip(lower=1)
df['payment_diff']        = df['total_payment'] - df['order_value']  # over/underpayment diff

# ── Customer satisfaction proxy ───────────────────────────────────────────────
df['review_is_positive']  = (df['review_score'] >= 4).astype(int)
df['review_is_negative']  = (df['review_score'] <= 2).astype(int)

# ── Approval speed ────────────────────────────────────────────────────────────
df['approval_hours'] = (df['order_approved_at'] - df['order_purchase_timestamp']).dt.total_seconds() / 3600
df['approval_hours'] = df['approval_hours'].clip(0, 72).fillna(df['approval_hours'].median())

print('Feature engineering done. New shape:', df.shape)

In [ ]:
# ── RFM: Recency, Frequency, Monetary (customer-level features) ───────────────
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg(
    recency   = ('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id', 'count'),
    monetary  = ('order_value', 'sum')
).reset_index()

# Drop RFM columns if they already exist (makes cell safe to re-run)
rfm_cols = ['recency', 'frequency', 'monetary',
            'recency_score', 'frequency_score', 'monetary_score', 'rfm_score']
df = df.drop(columns=[c for c in rfm_cols if c in df.columns])

# Merge RFM back
df = df.merge(rfm, on='customer_id', how='left')

# RFM score (quintile-based)
# Use rank(method='first') to break ties so pd.qcut always produces exactly 5 bins
for col in ['recency', 'frequency', 'monetary']:
    df[f'{col}_score'] = pd.qcut(df[col].rank(method='first'), q=5, labels=[1,2,3,4,5])
df['rfm_score'] = df[['recency_score','frequency_score','monetary_score']].astype(float).sum(axis=1)

print('RFM features added. Shape:', df.shape)
print(df[['recency','frequency','monetary','rfm_score']].describe().round(2))

---
## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ── Target distribution: order_value (regression) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df['order_value'], bins=60, color='#4878cf', edgecolor='white', alpha=0.85)
axes[0].set_title('Order Value Distribution (raw)')
axes[0].set_xlabel('Order Value (BRL)')

axes[1].hist(np.log1p(df['order_value']), bins=60, color='#6acc65', edgecolor='white', alpha=0.85)
axes[1].set_title('Log(Order Value) Distribution')
axes[1].set_xlabel('log(1 + Order Value)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Target distribution: late_delivery (classification) ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['late_delivery'].value_counts()
axes[0].bar(['On Time', 'Late'], counts.values, color=['#6acc65','#d65f5f'], width=0.5)
axes[0].set_title('Late Delivery Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center')

# Late delivery by state
late_by_state = df.groupby('customer_state')['late_delivery'].mean().sort_values(ascending=False).head(15)
late_by_state.plot(kind='bar', ax=axes[1], color='#d65f5f', alpha=0.8)
axes[1].set_title('Late Delivery Rate by Customer State (top 15)')
axes[1].set_ylabel('Late Delivery Rate')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Top product categories ────────────────────────────────────────────────────
top_cats = df['main_category'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(14, 5))
top_cats.plot(kind='bar', ax=ax, color='#4878cf', alpha=0.85)
ax.set_title('Top 20 Product Categories by Order Count')
ax.set_ylabel('Orders')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Order trends over time ────────────────────────────────────────────────────
monthly = df.groupby(df['order_purchase_timestamp'].dt.to_period('M')).agg(
    orders     = ('order_id', 'count'),
    avg_value  = ('order_value', 'mean')
).reset_index()
monthly['order_purchase_timestamp'] = monthly['order_purchase_timestamp'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(16, 7))
axes[0].plot(monthly['order_purchase_timestamp'], monthly['orders'], marker='o', ms=3, color='#4878cf')
axes[0].set_title('Monthly Order Volume')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['order_purchase_timestamp'], monthly['avg_value'], marker='o', ms=3, color='#6acc65')
axes[1].set_title('Monthly Average Order Value (BRL)')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr_cols = ['order_value','num_items','total_freight','max_installments',
             'delivery_days_actual','delivery_delay_days','late_delivery',
             'review_score','approval_hours','freight_ratio','rfm_score']
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

---
## 5. Prepare Feature Matrix
### Define features, encode categoricals, build pipelines

In [ ]:
# ── Drop rows with any remaining nulls in model features ──────────────────────
NUMERIC_FEATURES = [
    'num_items', 'total_price', 'total_freight', 'num_sellers',
    'total_payment', 'max_installments', 'avg_weight_g',
    'review_score', 'has_review_comment', 'delivery_days_estimated',
    'purchase_hour', 'purchase_dayofweek', 'purchase_month', 'purchase_quarter',
    'is_weekend', 'is_business_hours', 'freight_ratio', 'avg_item_price',
    'approval_hours', 'recency', 'frequency', 'monetary', 'rfm_score'
]
CATEGORICAL_FEATURES = ['customer_state', 'main_category', 'primary_payment']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

REG_TARGET  = 'order_value'
CLF_TARGET  = 'late_delivery'

# Drop rows where either target is missing
model_df = df[ALL_FEATURES + [REG_TARGET, CLF_TARGET] + ['delivery_days_actual']].dropna()
# Remove orders with no delivery date (can't compute delay)
model_df = model_df[model_df['delivery_days_actual'] > 0].drop(columns='delivery_days_actual')

print(f'Model-ready rows: {model_df.shape[0]:,}')
print(f'Class balance — late: {model_df[CLF_TARGET].mean():.2%}')

X = model_df[ALL_FEATURES]
y_reg = np.log1p(model_df[REG_TARGET])   # log-transform for regression
y_clf = model_df[CLF_TARGET]

In [ ]:
# ── Preprocessing pipeline ────────────────────────────────────────────────────
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES)
])

# Train/test splits
X_train, X_test, yr_train, yr_test = train_test_split(X, y_reg, test_size=0.2, random_state=SEED)
_, _,          yc_train, yc_test   = train_test_split(X, y_clf, test_size=0.2, random_state=SEED)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

---
## 6. Regression Models — Predict Order Value

In [ ]:
def evaluate_regressor(name, model, X_tr, y_tr, X_te, y_te):
    """Fit, predict, and return metrics. Inverse-transforms log target."""
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_tr, y_tr)
    preds_log = pipe.predict(X_te)
    preds = np.expm1(preds_log)
    actuals = np.expm1(y_te)
    mae  = mean_absolute_error(actuals, preds)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    r2   = r2_score(actuals, preds)
    cv   = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f'{name:35s} MAE={mae:7.2f}  RMSE={rmse:8.2f}  R²={r2:.4f}  CV-R²={cv:.4f}')
    return {'model': name, 'pipe': pipe, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'CV_R2': cv}

reg_results = []
reg_configs = [
    ('Linear Regression',          LinearRegression()),
    ('Ridge (alpha=1)',            Ridge(alpha=1.0)),
    ('Lasso (alpha=0.01)',         Lasso(alpha=0.01, max_iter=5000)),
    ('ElasticNet',                 ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000)),
    ('Decision Tree',              DecisionTreeRegressor(max_depth=8, random_state=SEED)),
    ('Random Forest',              RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1)),
    ('Gradient Boosting (sklearn)',GradientBoostingRegressor(n_estimators=200, random_state=SEED)),
    ('XGBoost',                    XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0)),
    ('LightGBM',                   LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=SEED, verbose=-1)),
    ('CatBoost',                   CatBoostRegressor(iterations=300, learning_rate=0.05, random_seed=SEED, verbose=0)),
]

print(f'{"Model":35s} MAE       RMSE      R²      CV-R²')
print('-' * 80)
for name, model in reg_configs:
    result = evaluate_regressor(name, model, X_train, yr_train, X_test, yr_test)
    reg_results.append(result)

reg_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'pipe'} for r in reg_results])
print('\n=== REGRESSION LEADERBOARD ===')
print(reg_df.sort_values('R2', ascending=False).to_string(index=False))

In [ ]:
# ── Regression comparison chart ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sorted_df = reg_df.sort_values('R2')

axes[0].barh(sorted_df['model'], sorted_df['R2'], color='#4878cf', alpha=0.85)
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score — All Regression Models')
axes[0].axvline(0, color='gray', lw=0.5)

axes[1].barh(sorted_df['model'], sorted_df['MAE'], color='#d65f5f', alpha=0.85)
axes[1].set_xlabel('MAE (BRL)')
axes[1].set_title('Mean Absolute Error — All Regression Models')
plt.tight_layout()
plt.show()

In [ ]:
# ── Actual vs Predicted — Best model ─────────────────────────────────────────
best_reg = sorted(reg_results, key=lambda x: x['R2'], reverse=True)[0]
preds_best = np.expm1(best_reg['pipe'].predict(X_test))
actuals    = np.expm1(yr_test)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(actuals, preds_best, alpha=0.3, s=10, color='#4878cf')
lims = [min(actuals.min(), preds_best.min()), max(actuals.max(), preds_best.max())]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Order Value (BRL)')
ax.set_ylabel('Predicted Order Value (BRL)')
ax.set_title(f'Actual vs Predicted — {best_reg["model"]}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Classification Models — Predict Late Delivery

In [ ]:
def evaluate_classifier(name, model, X_tr, y_tr, X_te, y_te):
    pipe = Pipeline([('pre', preprocessor), ('model', model)])
    pipe.fit(X_tr, y_tr)
    preds = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else None
    acc   = accuracy_score(y_te, preds)
    auc   = roc_auc_score(y_te, proba) if proba is not None else float('nan')
    cv    = cross_val_score(pipe, X_tr, y_tr, cv=5, scoring='roc_auc').mean()
    print(f'{name:35s} Acc={acc:.4f}  AUC={auc:.4f}  CV-AUC={cv:.4f}')
    return {'model': name, 'pipe': pipe, 'Accuracy': acc, 'AUC': auc, 'CV_AUC': cv, 'preds': preds, 'proba': proba}

clf_results = []
clf_configs = [
    ('Logistic Regression',        LogisticRegression(max_iter=500, random_state=SEED)),
    ('Decision Tree',              DecisionTreeClassifier(max_depth=8, random_state=SEED)),
    ('Random Forest',              RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
    ('Gradient Boosting',          GradientBoostingClassifier(n_estimators=200, random_state=SEED)),
    ('SVM (RBF kernel)',           SVC(probability=True, kernel='rbf', random_state=SEED)),
    ('K-Nearest Neighbors',        KNeighborsClassifier(n_neighbors=11, n_jobs=-1)),
    ('Naive Bayes',                GaussianNB()),
    ('XGBoost',                    XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                                   use_label_encoder=False, eval_metric='logloss',
                                                   random_state=SEED, verbosity=0)),
    ('LightGBM',                   LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=SEED, verbose=-1)),
    ('CatBoost',                   CatBoostClassifier(iterations=300, learning_rate=0.05, random_seed=SEED, verbose=0)),
]

print(f'{"Model":35s} Accuracy   AUC      CV-AUC')
print('-' * 70)
for name, model in clf_configs:
    result = evaluate_classifier(name, model, X_train, yc_train, X_test, yc_test)
    clf_results.append(result)

clf_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['pipe','preds','proba']} for r in clf_results])
print('\n=== CLASSIFICATION LEADERBOARD ===')
print(clf_df.sort_values('AUC', ascending=False).to_string(index=False))

In [ ]:
# ── ROC curves — all classifiers ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10.colors
for i, r in enumerate(clf_results):
    if r['proba'] is not None:
        fpr, tpr, _ = roc_curve(yc_test, r['proba'])
        ax.plot(fpr, tpr, lw=1.5, color=colors[i % 10], label=f"{r['model']} (AUC={r['AUC']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=0.8)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Classifiers')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion matrix — best classifier ───────────────────────────────────────
best_clf = sorted(clf_results, key=lambda x: x['AUC'], reverse=True)[0]
cm = confusion_matrix(yc_test, best_clf['preds'])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['On Time','Late']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_clf["model"]}')
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(yc_test, best_clf['preds'], target_names=['On Time','Late']))

---
## 8. Feature Importance & SHAP Analysis

In [ ]:
# ── Feature importance from best tree-based regressor ────────────────────────
best_tree_reg = next(r for r in sorted(reg_results, key=lambda x: x['R2'], reverse=True)
                     if hasattr(r['pipe'].named_steps['model'], 'feature_importances_'))

fitted_pre = best_tree_reg['pipe'].named_steps['pre']
fitted_pre.fit(X_train, yr_train)  # ensure fitted
cat_feature_names = fitted_pre.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(CATEGORICAL_FEATURES)
all_feature_names = NUMERIC_FEATURES + list(cat_feature_names)

importances = best_tree_reg['pipe'].named_steps['model'].feature_importances_
feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', ax=ax, color='#4878cf', alpha=0.85)
ax.invert_yaxis()
ax.set_title(f'Top 20 Feature Importances — {best_tree_reg["model"]}')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP values — best classifier ────────────────────────────────────────────
# Use a sample for speed
SHAP_SAMPLE = 500

# Get preprocessed data
best_clf_pipe = best_clf['pipe']
X_test_transformed = best_clf_pipe.named_steps['pre'].transform(X_test.iloc[:SHAP_SAMPLE])
clf_model = best_clf_pipe.named_steps['model']

explainer = shap.TreeExplainer(clf_model)
shap_values = explainer.shap_values(X_test_transformed)

# If binary, take positive class
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap.summary_plot(sv, X_test_transformed,
                  feature_names=all_feature_names,
                  max_display=15, show=True,
                  plot_title=f'SHAP Summary — {best_clf["model"]}')

---
## 9. Hyperparameter Tuning — Best Model

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Tune XGBoost Classifier (best or near-best model)
param_grid = {
    'model__n_estimators':    [200, 300, 400],
    'model__max_depth':       [4, 6, 8],
    'model__learning_rate':   [0.01, 0.05, 0.1],
    'model__subsample':       [0.7, 0.8, 1.0],
    'model__colsample_bytree':[0.7, 0.8, 1.0],
}

xgb_pipe = Pipeline([('pre', preprocessor),
                     ('model', XGBClassifier(use_label_encoder=False,
                                              eval_metric='logloss',
                                              random_state=SEED, verbosity=0))])

rscv = RandomizedSearchCV(
    xgb_pipe, param_grid,
    n_iter=20, scoring='roc_auc',
    cv=5, random_state=SEED, n_jobs=-1, verbose=1
)
rscv.fit(X_train, yc_train)

print(f'Best CV AUC: {rscv.best_score_:.4f}')
print(f'Best params: {rscv.best_params_}')

# Evaluate tuned model
tuned_preds = rscv.best_estimator_.predict(X_test)
tuned_proba = rscv.best_estimator_.predict_proba(X_test)[:, 1]
print(f'Tuned Test AUC: {roc_auc_score(yc_test, tuned_proba):.4f}')
print(f'Tuned Test Acc: {accuracy_score(yc_test, tuned_preds):.4f}')

---
## 10. Save Best Models & Summary

In [ ]:
import joblib, os
os.makedirs('./models', exist_ok=True)

# Save best regression pipeline
joblib.dump(best_reg['pipe'],  './models/best_regressor.pkl')

# Save tuned classification pipeline
joblib.dump(rscv.best_estimator_, './models/best_classifier.pkl')

# Save leaderboard CSVs
reg_df.sort_values('R2', ascending=False).to_csv('./models/regression_leaderboard.csv', index=False)
clf_df.sort_values('AUC', ascending=False).to_csv('./models/classification_leaderboard.csv', index=False)

print('Models saved to ./models/')
print('\n=== FINAL REGRESSION LEADERBOARD ===')
print(reg_df.sort_values('R2', ascending=False)[['model','MAE','RMSE','R2','CV_R2']].to_string(index=False))
print('\n=== FINAL CLASSIFICATION LEADERBOARD ===')
print(clf_df.sort_values('AUC', ascending=False)[['model','Accuracy','AUC','CV_AUC']].to_string(index=False))

---
## 11. Streamlit App — Quick Inference UI

Save the code below as `app/app.py` and run with `streamlit run app/app.py`

In [ ]:
streamlit_code = '''
# app/app.py — Olist ML Inference App
import streamlit as st
import pandas as pd
import numpy as np
import joblib

st.set_page_config(page_title="Olist ML Dashboard", layout="wide")
st.title("🛒 Olist E-Commerce — ML Predictions")
st.markdown("Predict **order value** and **late delivery risk** from order features.")

@st.cache_resource
def load_models():
    reg = joblib.load('../models/best_regressor.pkl')
    clf = joblib.load('../models/best_classifier.pkl')
    return reg, clf

reg_model, clf_model = load_models()

col1, col2 = st.columns(2)
with col1:
    st.subheader("Order Details")
    num_items         = st.slider("Number of items", 1, 20, 2)
    total_price       = st.number_input("Total price (BRL)", 10.0, 2000.0, 150.0)
    total_freight     = st.number_input("Freight value (BRL)", 5.0, 200.0, 20.0)
    max_installments  = st.slider("Max installments", 1, 24, 1)
    review_score      = st.slider("Expected review score", 1, 5, 4)
    main_category     = st.selectbox("Product category", ['bed_bath_table','health_beauty','sports_leisure','furniture_decor','computers_accessories'])
    primary_payment   = st.selectbox("Payment type", ['credit_card','boleto','debit_card','voucher'])
    customer_state    = st.selectbox("Customer state", ['SP','RJ','MG','RS','PR'])

with col2:
    st.subheader("Temporal / Logistics")
    purchase_hour     = st.slider("Purchase hour", 0, 23, 14)
    purchase_month    = st.slider("Purchase month", 1, 12, 6)
    delivery_days_est = st.slider("Estimated delivery days", 1, 60, 15)
    approval_hours    = st.slider("Approval time (hours)", 0.0, 48.0, 1.5)

if st.button("Predict", type="primary"):
    input_df = pd.DataFrame([dict(
        num_items=num_items, total_price=total_price, total_freight=total_freight,
        num_sellers=1, total_payment=total_price+total_freight,
        max_installments=max_installments, avg_weight_g=500,
        review_score=review_score, has_review_comment=0,
        delivery_days_estimated=delivery_days_est,
        purchase_hour=purchase_hour, purchase_dayofweek=2,
        purchase_month=purchase_month, purchase_quarter=(purchase_month-1)//3+1,
        is_weekend=0, is_business_hours=int(9<=purchase_hour<=18),
        freight_ratio=total_freight/(total_price+1),
        avg_item_price=total_price/max(num_items,1),
        payment_diff=0, review_is_positive=int(review_score>=4),
        review_is_negative=int(review_score<=2),
        approval_hours=approval_hours, recency=30, frequency=1, monetary=total_price,
        rfm_score=9, recency_score=3, frequency_score=3, monetary_score=3,
        customer_state=customer_state, main_category=main_category,
        primary_payment=primary_payment
    )])
    pred_value = np.expm1(reg_model.predict(input_df)[0])
    pred_late  = clf_model.predict(input_df)[0]
    pred_prob  = clf_model.predict_proba(input_df)[0][1]

    st.divider()
    m1, m2, m3 = st.columns(3)
    m1.metric("Predicted Order Value", f"R$ {pred_value:.2f}")
    m2.metric("Late Delivery Risk",    f"{pred_prob*100:.1f}%")
    m3.metric("Delivery Prediction",  "⚠️ LATE" if pred_late else "✅ ON TIME")
'''

import os
os.makedirs('./app', exist_ok=True)
with open('./app/app.py', 'w') as f:
    f.write(streamlit_code.strip())
print('Streamlit app saved to ./app/app.py')
print('Run with: streamlit run app/app.py')

---
## 12. GitHub Repository Structure

```
ml-portfolio/
├── 01_classical_ml_olist.ipynb   ← this notebook
├── data/                          ← raw CSVs (gitignored if large)
├── models/
│   ├── best_regressor.pkl
│   ├── best_classifier.pkl
│   ├── regression_leaderboard.csv
│   └── classification_leaderboard.csv
├── app/
│   └── app.py                     ← Streamlit UI
├── requirements.txt
└── README.md
```

### requirements.txt
```
pandas numpy matplotlib seaborn scikit-learn
xgboost lightgbm catboost shap missingno plotly
streamlit joblib
```

### Deploy Streamlit app (free)
1. Push repo to GitHub
2. Go to https://share.streamlit.io
3. Connect your repo → select `app/app.py` → Deploy ✅